# Python for Scientific Computing
## Max Planck Computing and Data Facility, Garching

# Example: Heat Equation

* toy problem: (constant) diffusion on a 2d periodic grid $u_t = \alpha \Delta u$
* explicit time stepping
* **lecture example**
    * finite difference Laplacian operator, 5-point stencil
    * formula for time update of $u$ at position $x_i,y_j$ from time $t_n$ to $t_{n+1}$: 
    $$ u_{i,j}^{(n+1)} = u_{i,j}^{(n)} + \frac{\alpha \Delta t}{\Delta x^2} \left( 
    u_{i-1,j}^{(n)}
    +u_{i+1,j}^{(n)}
    +u_{i,j-1}^{(n)}
    +u_{i,j+1}^{(n)}
    -4u_{i,j}^{(n)}
    \right) $$
* **exercise**
    * use a 9-point stencil finite difference Laplacian operator
    * formula for time update of $u$ at position $x_i,y_j$ from time $t_n$ to $t_{n+1}$: 
    $$ u_{i,j}^{(n+1)} = u_{i,j}^{(n)} + \frac{\alpha \Delta t}{3 \Delta x^2} \left( 
      u_{i+1,j}^{(n)} + u_{i-1,j}^{(n)} + u_{i,j+1}^{(n)} + u_{i,j-1}^{(n)}
    + u_{i+1,j+1}^{(n)} + u_{i-1,j-1}^{(n)} + u_{i+1,j-1}^{(n)} + u_{i-1,j+1}^{(n)} - 8u_{i,j}^{(n)}
    \right) $$
* implemente various approaches
    * naive Python implementation
    * NumPy variants
    * Cython
    * Numba
* measure, compare, improve

In [1]:
# Technical detail: limit the number of threads to allow for a fair comparison,
#                   some NumPy expressions would try to parallelize automatically
import os
for VAR in ["OMP_NUM_THREADS", "NUMBA_NUM_THREADS"]:
    if VAR not in os.environ:
        os.environ[VAR]="2"
    print("{}={}".format(VAR, os.environ[VAR]))
import numpy as np

OMP_NUM_THREADS=2
NUMBA_NUM_THREADS=2


## Infrastructure code: parameters, init(), main_loop()

In [2]:
# parameters (global, for convenience)
n_iterations = 50
n_points = 254
dt = 0.12
D = 0.8

In [3]:
def init(val=0.5):
    """Set up a 2d NumPy array with some initial value pattern."""
    x = np.linspace(0., 4.*np.pi, num=n_points+2)
    y = np.linspace(0., 4.*np.pi, num=n_points+2)
    grid = val * np.outer(np.sin(x)**4, np.sin(y)**4)
    return grid

In [4]:
def main_loop(evolve_func, grid):
    """Main loop function, calling evolve_func on grid."""
    grid_tmp = np.empty_like(grid)
    for i in range(1, n_iterations+1):
        evolve_func(grid, grid_tmp, n_points, dt, D)
        # swap references, do not copy
        grid_foo = grid
        grid = grid_tmp
        grid_tmp = grid_foo
    return grid

## 1. A naive pure Python implementation

In [5]:
def apply_periodic_bc_python(grid, n_points):
    """Explicitly apply periodic boundary conditions, via Python loops."""
    for j in range(n_points + 2):
        grid[ 0, j] = grid[-2, j]
        grid[-1, j] = grid[ 1, j]
    for i in range(n_points + 2):
        grid[ i,-1] = grid[ i, 1]
        grid[ i, 0] = grid[ i,-2]

In [6]:
def evolve_python(grid, grid_tmp, n_points, dt, D):
    """Time-step using 5-pt stencil, as presented in the lecture."""
    apply_periodic_bc_python(grid, n_points)
    for i in range(1, n_points+1):
        for j in range(1, n_points+1):
            # stencil formula
            grid_tmp[i, j] = grid[i, j] + dt * D * (
                grid[i-1, j] + grid[i+1, j] + grid[i, j-1] + grid[i, j+1]
                - 4.0 * grid[i, j])

In [8]:
%%time
grid = init()
solution_python = main_loop(evolve_python, grid)

CPU times: total: 3.61 s
Wall time: 3.66 s


In [10]:
def evolve_python_nine(grid, grid_tmp, n_points, dt, D):
    """Time-step using 9-point stencil."""
    apply_periodic_bc_python(grid, n_points)

    for i in range(1, n_points + 1):
        for j in range(1, n_points + 1):
            grid_tmp[i, j] = grid[i, j] + (dt * D / 3.0) * (
                grid[i-1, j] +
                grid[i+1, j] +
                grid[i, j-1] +
                grid[i, j+1] +
                grid[i-1, j-1] +
                grid[i-1, j+1] +
                grid[i+1, j-1] +
                grid[i+1, j+1] -
                8.0 * grid[i, j]
            )

In [ ]:
%%time
grid = init()
solution_python_nine = main_loop(evolve_python_nine, grid)

In [10]:
import matplotlib.pyplot as plt

def plot_grids(grid_1, grid_2):
    f = plt.figure(figsize=(10,5))
    plt.subplot(121)
    plt.pcolormesh(grid_1.T)
    plt.axis('image')
    plt.title('grid_1')
    plt.subplot(122)
    plt.pcolormesh(grid_2.T)
    plt.axis('image')
    l = plt.title('grid_2')

plot_grids(solution_python, solution_python_nine)

In [11]:
# look at the deviation of the solutions between the 5pt and the 9pt stencil
rel = np.max(np.abs(solution_python - solution_python_nine))/np.max(np.abs(solution_python))
print(rel)

0.00102803880472


## 2. NumPy implementations

In [11]:
def evolve_numpy_slicing_nine(grid, grid_tmp, n_points, dt, D):
    """Time-step using 9-pt stencil."""
    apply_periodic_bc_python(grid, n_points)

    grid_tmp[1:-1, 1:-1] = grid[1:-1, 1:-1] + (dt * D / 3.0) * (
        grid[:-2, 1:-1] +
        grid[2:, 1:-1] +
        grid[1:-1, :-2] +
        grid[1:-1, 2:] +
        grid[:-2, :-2] +
        grid[:-2, 2:] +
        grid[2:, :-2] +
        grid[2:, 2:] -
        8.0 * grid[1:-1, 1:-1]
    )

In [12]:
%%time
grid = init()
solution_numpy_slicing_nine = main_loop(evolve_numpy_slicing_nine, grid)

CPU times: total: 62.5 ms
Wall time: 70.7 ms


In [ ]:
plot_grids(solution_python, solution_numpy_slicing_nine)

## 3. Cython implementation

In [29]:
%load_ext Cython

The Cython extension is already loaded. To reload it, use:
  %reload_ext Cython


In [34]:
%%cython

cimport cython

@cython.boundscheck(False)
@cython.wraparound(False)
def evolve_cython_nine(double[:, :] grid,
                       double[:, :] grid_tmp,
                       int n_points,
                       double dt,
                       double D):
    cdef int i, j

    # Condiciones periódicas
    for j in range(n_points + 2):
        grid[0, j] = grid[n_points, j]
        grid[n_points + 1, j] = grid[1, j]

    for i in range(n_points + 2):
        grid[i, 0] = grid[i, n_points]
        grid[i, n_points + 1] = grid[i, 1]

    # Laplaciano de 9 puntos
    for i in range(1, n_points + 1):
        for j in range(1, n_points + 1):
            grid_tmp[i, j] = grid[i, j] + (dt * D / 3.0) * (
                grid[i-1, j]
                + grid[i+1, j]
                + grid[i, j-1]
                + grid[i, j+1]
                + grid[i-1, j-1]
                + grid[i-1, j+1]
                + grid[i+1, j-1]
                + grid[i+1, j+1]
                - 8.0 * grid[i, j]
            )

PlatformError: Microsoft Visual C++ 14.0 or greater is required. Get it with "Microsoft C++ Build Tools": https://visualstudio.microsoft.com/visual-cpp-build-tools/

In [17]:
%%time
grid = init()
solution_cython_nine = main_loop(evolve_cython_nine, grid)

CPU times: user 14.7 ms, sys: 138 µs, total: 14.9 ms
Wall time: 64.8 ms


In [ ]:
plot_grids(solution_python, solution_cython_nine)

## 4. Numba implementation

In [35]:
from numba import jit

In [36]:
@jit(nopython=True)
def apply_periodic_bc_numba(grid, n_points):
    """Explicitly apply periodic boundary conditions."""
    for j in range(n_points + 2):
        grid[0, j] = grid[n_points, j]
        grid[n_points + 1, j] = grid[1, j]

    for i in range(n_points + 2):
        grid[i, 0] = grid[i, n_points]
        grid[i, n_points + 1] = grid[i, 1]

In [37]:
@jit(nopython=True)
def evolve_numba_nine(grid, grid_tmp, n_points, dt, D):
    """Time-step using 9-pt stencil."""
    apply_periodic_bc_numba(grid, n_points)

    for i in range(1, n_points + 1):
        for j in range(1, n_points + 1):
            grid_tmp[i, j] = grid[i, j] + (dt * D / 3.0) * (
                grid[i-1, j]
                + grid[i+1, j]
                + grid[i, j-1]
                + grid[i, j+1]
                + grid[i-1, j-1]
                + grid[i-1, j+1]
                + grid[i+1, j-1]
                + grid[i+1, j+1]
                - 8.0 * grid[i, j]
            )

In [38]:
# run once without timing in order to compile, do the timing below
grid = init()
solution_numba_nine = main_loop(evolve_numba_nine, grid)

In [39]:
%%time
grid = init()
solution_numba_nine = main_loop(evolve_numba_nine, grid)

CPU times: total: 0 ns
Wall time: 4.08 ms


In [ ]:
plot_grids(solution_python, solution_numba_nine)

## Optional exercise parts
* Further increase the order/stencil width of the Laplacian (requires to increase the width of the ghost cells for the periodic boundary conditions)
* Implement functions to write the grid every few time steps to a HDF5 file (and to read it in again)
* Implement a restart functionality: restart the simulation from a saved HDF5 file
* Implement reading in a parameter file in YAML format for the simulation parameters (number of iterations, number of points, timestep...)
* Complete the MPI implementation in `Diffusion_MPI.py` that uses a domain decomposition approach (complete communication of boundaries and communication of whole grid to root for plotting)